# Multi-Head Attention 多头注意力

Multi-Head Attention 是在 scaled dot-product attention 的基础上，把同一个输入投影到多个不同的子空间中，分别计算 attention，再把多个 head 的结果拼接起来并通过输出线性层融合。

直观来说，单头 attention 只会得到一张 attention map；而在复杂任务中，不同 token 之间可能存在多种关系。例如有的 head 更关注局部相邻 token，有的 head 更关注长距离依赖，有的 head 可能更偏向语法结构或语义相关性。Multi-Head Attention 让模型可以并行学习多组不同的注意力模式。

## 标准公式

设输入为：

$$
X \in \mathbb{R}^{B \times T \times d_{model}}
$$

其中 $B$ 是 batch size，$T$ 是 sequence length，$d_{model}$ 是 hidden dimension。假设有 $h$ 个 attention heads，那么每个 head 的维度通常为：

$$
d_{head} = \frac{d_{model}}{h}
$$

因此要求：

$$
d_{model} \bmod h = 0
$$

第 $i$ 个 head 会有自己的一组投影：

$$
Q_i = XW_i^Q, \quad K_i = XW_i^K, \quad V_i = XW_i^V
$$

然后计算：

$$
head_i = \operatorname{Attention}(Q_i, K_i, V_i)
$$

其中：

$$
\operatorname{Attention}(Q_i, K_i, V_i) = \operatorname{softmax}\left(\frac{Q_iK_i^T}{\sqrt{d_{head}}}\right)V_i
$$

最后把所有 head 的输出在 channel 维度拼接起来，再经过输出线性层：

$$
\operatorname{MultiHead}(X) = \operatorname{Concat}(head_1, head_2, \cdots, head_h)W^O
$$

## Shape 变化

在实现中，通常不会真的为每个 head 分别写一个独立的线性层，而是用一个大的线性层一次性生成 Q、K、V：

$$
QKV = XW_{qkv}
$$

对应代码中：

```python
self.W_qkv = nn.Linear(hidden_dim, hidden_dim * 3)
```

主要 shape 变化如下：

$$
X: [B, T, D]
$$

$$
QKV: [B, T, 3D]
$$

拆分 Q、K、V 后：

$$
Q,K,V: [B, T, D]
$$

再 reshape 成多头形式：

$$
Q,K,V: [B, h, T, d_{head}]
$$

注意力分数为：

$$
Scores = \frac{QK^T}{\sqrt{d_{head}}}
$$

因此：

$$
Scores: [B, h, T, T]
$$

每个 head 都会得到一张自己的 attention map。经过 softmax 和对 $V$ 加权求和后：

$$
Output_{heads}: [B, h, T, d_{head}]
$$

把 head 维度和 $d_{head}$ 维度重新拼接：

$$
Output_{concat}: [B, T, D]
$$

最后通过输出线性层 $W^O$ 做融合，输出 shape 仍然是：

$$
Output: [B, T, D]
$$

## Mask 在多头中的广播

如果 mask 的 shape 是：

$$
Mask: [B, T, T]
$$

它表示每个样本中 query token 能否看到 key token。由于 attention scores 的 shape 是：

$$
Scores: [B, h, T, T]
$$

所以 mask 需要在 head 维度上广播：

$$
Mask: [B, 1, T, T]
$$

这样同一个 mask 会应用到每一个 head 上。

本 notebook 下面的代码实现了一个简化版 Multi-Head Attention，重点展示 QKV 拆分、多头 reshape、mask 处理、head 拼接和输出线性融合的完整流程。


In [ ]:
# multi-head attention implementation
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.W_qkv = nn.Linear(hidden_dim, hidden_dim * 3)
        self.W_o = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()

        # x: [batch_size, seq_len, hidden_dim]
        qkv = self.W_qkv(x)  # [batch_size, seq_len, hidden_dim * 3]
        q, k, v = qkv.chunk(3, dim=-1)  # each: [batch_size, seq_len, hidden_dim]

        # [batch_size, seq_len, hidden_dim] -> [batch_size, num_heads, seq_len, head_dim]
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # [batch_size, num_heads, seq_len, seq_len]
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            # mask can be [seq_len, seq_len], [batch_size, seq_len, seq_len],
            # or [batch_size, 1, seq_len, seq_len]. It is broadcast across heads.
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            elif mask.dim() != 4:
                raise ValueError("mask must have shape [T,T], [B,T,T], or [B,1,T,T]")
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)  # [batch_size, num_heads, seq_len, seq_len]

        # [batch_size, num_heads, seq_len, head_dim]
        output_heads = torch.matmul(attention_weights, v)

        # [batch_size, num_heads, seq_len, head_dim] -> [batch_size, seq_len, hidden_dim]
        output_concat = output_heads.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)
        output = self.W_o(output_concat)

        return output, attention_weights


/home/yanrui/anaconda3/envs/py38/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# test multi-head attention
x = torch.rand(2, 5, 16)  # batch_size=2, seq_len=5, hidden_dim=16
mask = torch.tril(torch.ones(5, 5)).unsqueeze(0).repeat(2, 1, 1)

print("mask shape:", mask.shape)

multihead_attention = MultiHeadAttention(hidden_dim=16, num_heads=4)
output, attention_weights = multihead_attention(x, mask)
print("output shape:", output.shape)  # should be [2, 5, 16]
print("attention_weights shape:", attention_weights.shape)  # should be [2, 4, 5, 5]


torch.Size([2, 5, 5])


RuntimeError: The size of tensor a (2) must match the size of tensor b (4) at non-singleton dimension 1